# Cybersecurity Analytics & Intrusion Detection Dashboard
## Notebook 3 — Démo d'inférence

Montre comment **utiliser le modèle sauvegardé** (`models/xgb_model.json`) sur de nouveaux flux réseau : chargement, nettoyage, prédiction, alertes.



In [1]:
import json
import time

import numpy as np
import pandas as pd
import xgboost as xgb

MODELS_DIR = "../models/"
PROCESSED_CSV = "../data/processed/cleaned_network_traffic.csv"

with open(MODELS_DIR + "meta.json", encoding="utf-8") as f:
    meta = json.load(f)

model = xgb.XGBClassifier()
model.load_model(MODELS_DIR + "xgb_model.json")

FEATURES = meta["features"]
CLASSES = meta["classes"]
print("Modèle chargé :", meta["model_name"])
print("Features :", len(FEATURES), "| Classes :", CLASSES)

Modèle chargé : XGBoost (poids sqrt)
Features : 79 | Classes : ['Benign', 'Bot', 'Brute Force', 'DDoS', 'DoS', 'Port Scan', 'Web Attack']


## 1. Fonction de prédiction

In [2]:
def predict_flows(flows: pd.DataFrame, alert_threshold: float = 0.5) -> pd.DataFrame:
    """
    flows : DataFrame contenant (au moins) les colonnes de meta["features"].
    Retourne le DataFrame avec : Predicted_Category, Confidence, Is_Alert.
    """
    missing = [c for c in FEATURES if c not in flows.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing[:5]}{'...' if len(missing) > 5 else ''}")

    X = flows[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0).astype("float32")
    proba = model.predict_proba(X)

    out = flows.copy()
    out["Predicted_Category"] = [CLASSES[i] for i in proba.argmax(axis=1)]
    out["Confidence"] = proba.max(axis=1).round(4)
    # Alerte = probabilité cumulée de "non-Benign" au-dessus du seuil
    benign_idx = CLASSES.index(meta["benign_class"])
    out["Attack_Probability"] = (1 - proba[:, benign_idx]).round(4)
    out["Is_Alert"] = out["Attack_Probability"] >= alert_threshold
    return out

print("Fonction prête.")

Fonction prête.


## 2. Essai sur un échantillon (~1 % du dataset)

In [3]:
rng = np.random.default_rng(42)
sample = pd.read_csv(PROCESSED_CSV, skiprows=lambda i: i > 0 and rng.random() > 0.01, low_memory=False)
sample = sample[sample["Attack_Category"].isin(CLASSES)]
print("Échantillon :", sample.shape)

t0 = time.time()
scored = predict_flows(sample)
elapsed = time.time() - t0
print(f"{len(scored):,} flux scorés en {elapsed:.2f} s ({elapsed / len(scored) * 1e6:.1f} µs / flux)")

scored[["Attack_Category", "Predicted_Category", "Confidence", "Attack_Probability", "Is_Alert"]].head(10)

Échantillon : (25011, 85)
25,011 flux scorés en 0.14 s (5.6 µs / flux)


,Attack_Category,Predicted_Category,Confidence,Attack_Probability,Is_Alert
0,Benign,Benign,1.0,0.0,False
1,Benign,Benign,1.0,0.0,False
2,Benign,Benign,1.0,0.0,False
3,Benign,Benign,1.0,0.0,False
4,Benign,Benign,1.0,0.0,False
5,Benign,Benign,1.0,0.0,False
6,Benign,Benign,1.0,0.0,False
7,Benign,Benign,1.0,0.0,False
8,Benign,Benign,1.0,0.0,False
9,Benign,Benign,1.0,0.0,False


In [4]:
print("Alertes générées :", int(scored["Is_Alert"].sum()), "sur", len(scored), "flux")
print()
print(pd.crosstab(scored["Attack_Category"], scored["Predicted_Category"]))

Alertes générées : 4268 sur 25011 flux

Predicted_Category  Benign  Bot  Brute Force  DDoS   DoS  Port Scan  \
Attack_Category                                                       
Benign               20739    7            0     1     4          7   
Bot                      2   14            0     0     0          0   
Brute Force              0    0           94     0     0          0   
DDoS                     0    0            0  1340     0          0   
DoS                      2    0            0     0  1874          7   
Port Scan                0    0            0     0     0        906   
Web Attack               0    0            0     0     0          0   

Predicted_Category  Web Attack  
Attack_Category                 
Benign                       0  
Bot                          0  
Brute Force                  0  
DDoS                         0  
DoS                          0  
Port Scan                    0  
Web Attack                  14  


## 3. Seuil d'alerte

`alert_threshold` règle le compromis **détection ↔ fausses alertes** : un seuil bas détecte plus d'attaques mais génère plus de fausses alertes ; un seuil haut fait l'inverse. En production, on le choisit selon la tolérance de l'équipe sécurité aux fausses alertes.

In [5]:
rows = []
is_attack = (scored["Attack_Category"] != "Benign")
for thr in [0.2, 0.5, 0.8, 0.95]:
    alert = scored["Attack_Probability"] >= thr
    rows.append({
        "Seuil": thr,
        "Attaques détectées (%)": round(alert[is_attack].mean() * 100, 2),
        "Fausses alertes (%)": round(alert[~is_attack].mean() * 100, 3),
    })
pd.DataFrame(rows)

,Seuil,Attaques détectées (%),Fausses alertes (%)
0,0.20,99.95,0.106
1,0.50,99.91,0.092
2,0.80,99.69,0.043
3,0.95,99.37,0.034
